# Семинар 5. Облака точек и графовые нейронные сети

На семинаре разберем две задачи.

1. Классификация облака точек. Требуется построить функцию множества, результат которой не зависит от порядка точек.
2. Классификация вершин графа. Требуется проверить, дает ли связность дополнительную информацию по сравнению с признаками вершин.

Сначала поработаем с S3DIS и небольшой синтетической задачей для PointNet, затем перейдем к графу Tolokers. kNN-граф нужен только как пример другого представления облака точек.


## Подготовка окружения

При первом запуске понадобятся доступ к сети и пакеты `torch_geometric` и `wldhx.yadisk-direct`. Установка выполняется только для отсутствующих пакетов.


In [ ]:
import importlib.util
import shutil
import subprocess
import sys

packages = []
if importlib.util.find_spec("torch") is None:
    packages.append("torch")
if importlib.util.find_spec("torch_geometric") is None:
    packages.append("torch-geometric")
if importlib.util.find_spec("sklearn") is None:
    packages.append("scikit-learn")
if shutil.which("yadisk-direct") is None:
    packages.append("wldhx.yadisk-direct")

if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


In [ ]:
import random
import subprocess
import tarfile
import time
import urllib.request
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch
from sklearn.metrics import roc_auc_score
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings(
    "ignore", category=FutureWarning, module=r"torch\.jit\._script"
)


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. Облака точек и вокселизация

### S3DIS

S3DIS содержит сканы помещений. Каждая точка задается координатами, цветом и меткой одного из 13 семантических классов. Полная задача состоит в классификации каждой точки, но здесь S3DIS используется для разбора представления данных и предобработки.

Ниже загружается уменьшенная версия одной области и визуализируется одна комната.


In [ ]:
USE_SMALL_S3DIS_SAMPLE = True

if USE_SMALL_S3DIS_SAMPLE:
    link_to_s3dis = "https://disk.yandex.ru/d/WpgCyMBKLfXQEQ"
    archive_name = "s3dis-compressed-sanity-check.zip"
else:
    link_to_s3dis = "https://disk.yandex.ru/d/4FWuY3WUBjIgJA"
    archive_name = "s3dis-compressed.tar.gz"

path_to_data_dir = archive_name.replace(".zip", "").replace(".tar.gz", "")

if not Path(path_to_data_dir).exists():
    direct_url = subprocess.run(
        ["yadisk-direct", link_to_s3dis],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    urllib.request.urlretrieve(direct_url, archive_name)

    if archive_name.endswith(".zip"):
        with zipfile.ZipFile(archive_name) as archive:
            archive.extractall()
    else:
        with tarfile.open(archive_name, "r:gz") as archive:
            archive.extractall()


In [ ]:
CLASS_NAMES = [
    "ceiling", "floor", "wall", "beam", "column", "window", "door",
    "table", "chair", "sofa", "bookcase", "board", "clutter",
]
COLORS = [
    "#0eeee7", "#fefe08", "#ff6b6b", "#ffa600", "#ffca3a", "#8ac926",
    "#1982c4", "#6a4c93", "#e36df5", "#00ffa8", "#ffd1dc", "#3cf0ff",
    "#ff4d00",
]

room_dir = Path(path_to_data_dir) / "Area_1" / "conferenceRoom_1"
coords = np.load(room_dir / "coord.npy")
labels = np.load(room_dir / "segment.npy").reshape(-1)


def show_cloud(coords, point_colors, title="", max_points=10_000):
    coords = np.asarray(coords)
    point_colors = np.asarray(point_colors)
    if coords.shape[0] > max_points:
        idx = np.random.default_rng(0).choice(
            coords.shape[0], max_points, replace=False
        )
        coords = coords[idx]
        point_colors = point_colors[idx]

    fig = go.Figure(data=[go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers",
        marker=dict(size=1.2, color=point_colors),
    )])
    fig.update_layout(
        title=title,
        scene_aspectmode="data",
        margin=dict(l=0, r=0, t=30, b=0),
        height=550,
    )
    fig.show()


print("точек в комнате:", coords.shape[0])
show_cloud(
    coords,
    np.asarray(COLORS)[labels],
    "S3DIS: конференц-зал, цвет соответствует классу точки",
)


### Вокселизация

Плотность точек в скане неравномерна. При вокселизации пространство разбивается на кубические ячейки со стороной `grid_size`, после чего из каждой занятой ячейки остается одна точка.

Как изменятся число точек и доля вокселей со смешанными семантическими метками при увеличении `grid_size`? Сначала запишите ответ, затем проверьте его по таблице.

Задание 1. Реализуйте `grid_sample`. Функция должна вернуть индексы одной точки из каждого занятого вокселя.


In [ ]:
def grid_sample(coords: np.ndarray, grid_size: float) -> np.ndarray:
    grid_coord = ...
    keep_idx = ...
    return keep_idx



toy_coords = np.array([
    [0.01, 0.01, 0.01],
    [0.04, 0.02, 0.03],
    [0.11, 0.01, 0.01],
    [0.12, 0.04, 0.03],
    [0.25, 0.01, 0.01],
])
toy_keep = grid_sample(toy_coords, grid_size=0.10)
toy_before = np.unique(np.floor(toy_coords / 0.10).astype(np.int64), axis=0)
toy_after = np.floor(toy_coords[toy_keep] / 0.10).astype(np.int64)
assert len(toy_keep) == len(toy_before)
assert len(np.unique(toy_after, axis=0)) == len(toy_after)


In [ ]:
def voxel_statistics(coords, labels, grid_size):
    voxel = np.floor(coords / grid_size).astype(np.int64)
    _, inverse = np.unique(voxel, axis=0, return_inverse=True)
    n_voxels = int(inverse.max()) + 1

    voxel_label_pairs = np.unique(
        np.column_stack([inverse, labels]), axis=0
    )
    labels_per_voxel = np.bincount(
        voxel_label_pairs[:, 0], minlength=n_voxels
    )
    mixed_fraction = float((labels_per_voxel > 1).mean())
    return n_voxels, mixed_fraction


print(f'{"grid":>8} {"точек":>10} {"осталось":>10} {"смешанные воксели":>20}')
for grid_size in [0.02, 0.05, 0.10]:
    keep_idx = grid_sample(coords, grid_size)
    n_voxels, mixed_fraction = voxel_statistics(coords, labels, grid_size)
    print(
        f'{grid_size:8.2f} {len(keep_idx):10d} '
        f'{len(keep_idx) / len(coords):10.3f} {mixed_fraction:20.3f}'
    )
    assert len(keep_idx) == n_voxels

keep = grid_sample(coords, 0.05)
show_cloud(
    coords[keep],
    np.asarray(COLORS)[labels[keep]],
    "S3DIS после вокселизации с шагом 5 см",
)


При увеличении `grid_size` точек остается меньше, а разные семантические классы чаще попадают в один воксель. По таблице выберите шаг, при котором облако уже заметно сжалось, но границы объектов еще не потерялись.

### Геометрические преобразования

Для помещений допустимы повороты вокруг вертикальной оси, небольшой шум координат и случайное прореживание. Поворот вокруг горизонтальной оси меняет направление пола и стен и обычно не сохраняет смысл сцены.


In [ ]:
def rotate_z(points: np.ndarray, angle: float) -> np.ndarray:
    rotation = np.array([
        [np.cos(angle), -np.sin(angle), 0.0],
        [np.sin(angle),  np.cos(angle), 0.0],
        [0.0,            0.0,           1.0],
    ])
    center = points.mean(axis=0, keepdims=True)
    return (points - center) @ rotation.T + center


rng = np.random.default_rng(42)
base = coords[keep]
augmented = rotate_z(base, angle=rng.uniform(-np.pi, np.pi))
augmented = augmented + rng.normal(0.0, 0.005, size=augmented.shape)

show_cloud(
    augmented,
    np.asarray(COLORS)[labels[keep]],
    "Поворот вокруг z и шум координат",
)


## 2. PointNet

Обучение сегментации на S3DIS занимает слишком много времени для одной пары. Поэтому PointNet обучим различать четыре синтетических объекта: сферу, куб, цилиндр и тор.

Каждое облако центрируется и нормируется на единичный шар. Поэтому классификатор не может решить задачу только по положению или общему масштабу объекта.


In [ ]:
N_POINTS = 256
SHAPE_NAMES = ["сфера", "куб", "цилиндр", "тор"]


def sample_sphere(n, rng):
    points = rng.normal(size=(n, 3))
    return points / np.linalg.norm(points, axis=1, keepdims=True)


def sample_cube(n, rng):
    points = rng.uniform(-1.0, 1.0, size=(n, 3))
    face = rng.integers(0, 3, size=n)
    sign = rng.choice([-1.0, 1.0], size=n)
    points[np.arange(n), face] = sign
    return points


def sample_cylinder(n, rng):
    angle = rng.uniform(0.0, 2.0 * np.pi, size=n)
    z = rng.uniform(-1.0, 1.0, size=n)
    return np.stack([np.cos(angle), np.sin(angle), z], axis=1)


def sample_torus(n, rng, major_radius=1.0, minor_radius=0.4):
    u = rng.uniform(0.0, 2.0 * np.pi, size=n)
    v = rng.uniform(0.0, 2.0 * np.pi, size=n)
    x = (major_radius + minor_radius * np.cos(v)) * np.cos(u)
    y = (major_radius + minor_radius * np.cos(v)) * np.sin(u)
    z = minor_radius * np.sin(v)
    return np.stack([x, y, z], axis=1)


SHAPE_SAMPLERS = [sample_sphere, sample_cube, sample_cylinder, sample_torus]


def normalize_cloud(points):
    points = points - points.mean(axis=0, keepdims=True)
    radius = np.linalg.norm(points, axis=1).max()
    return points / max(radius, 1e-12)


class FixedShapesDataset(Dataset):
    def __init__(self, size: int, seed: int):
        rng = np.random.default_rng(seed)
        clouds, labels = [], []
        for index in range(size):
            label = index % len(SHAPE_SAMPLERS)
            points = SHAPE_SAMPLERS[label](N_POINTS, rng)
            points = rotate_z(points, angle=rng.uniform(-np.pi, np.pi))
            points = points + rng.normal(0.0, 0.02, size=points.shape)
            points = normalize_cloud(points)
            clouds.append(points.astype(np.float32))
            labels.append(label)

        self.clouds = torch.from_numpy(np.stack(clouds))
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.clouds[index], self.labels[index]


train_shapes = FixedShapesDataset(1536, seed=0)
val_shapes = FixedShapesDataset(384, seed=1)
test_shapes = FixedShapesDataset(512, seed=2)

first_a, label_a = test_shapes[0]
first_b, label_b = test_shapes[0]
assert torch.equal(first_a, first_b) and label_a == label_b
assert torch.allclose(first_a.mean(0), torch.zeros(3), atol=1e-5)
assert torch.isclose(first_a.norm(dim=1).max(), torch.tensor(1.0), atol=1e-5)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), subplot_kw={"projection": "3d"})
for label, axis in enumerate(axes):
    points, _ = train_shapes[label]
    axis.scatter(points[:, 0], points[:, 1], points[:, 2], s=3)
    axis.set_title(SHAPE_NAMES[label])
    axis.set_axis_off()
plt.show()


### Инвариантность к перестановке точек

Для облака точек порядок элементов не несет информации. PointNet применяет один и тот же MLP к каждой точке, а затем использует симметричную агрегацию по множеству точек.

$$
f(\{x_1, \ldots, x_n\}) = \rho\!\left(\max_{i=1,\ldots,n} \phi(x_i)\right).
$$

Задание 2. Допишите две строки в `forward`: поточечное преобразование и max pooling по размерности точек.


In [ ]:
class TinyPointNet(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.point_mlp = nn.Sequential(
            nn.Linear(3, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, points):
        features = ...
        pooled = ...
        return self.head(pooled)



set_global_seed(42)
pointnet = TinyPointNet().to(device)

points, _ = train_shapes[0]
points = points.unsqueeze(0).to(device)
permutation = torch.randperm(points.shape[1], device=device)

with torch.no_grad():
    logits = pointnet(points)
    permuted_logits = pointnet(points[:, permutation])
    point_features = pointnet.point_mlp(points)
    order_sensitive = point_features[:, 0]
    order_sensitive_permuted = point_features[:, permutation][:, 0]

assert logits.shape == (1, 4)
assert torch.allclose(logits, permuted_logits, atol=1e-6)
print("max pooling, изменение выхода:", float((logits - permuted_logits).abs().max()))
print(
    "первая точка вместо pooling, изменение представления:",
    float((order_sensitive - order_sensitive_permuted).norm()),
)


Max pooling дает точную инвариантность к перестановке. Выбор первой точки в качестве представления множества этим свойством не обладает.

### Обучение на синтетической задаче

Для выбора эпохи используется validation accuracy. Test используется один раз после восстановления лучшего состояния модели.


In [ ]:
def classification_accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for points, labels in loader:
            points = points.to(device)
            labels = labels.to(device)
            predictions = model(points).argmax(dim=1)
            correct += int((predictions == labels).sum())
            total += labels.numel()
    return correct / total


def fit_pointnet(model, max_epochs=10, patience=3):
    train_loader = DataLoader(train_shapes, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_shapes, batch_size=128)
    test_loader = DataLoader(test_shapes, batch_size=128)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = -np.inf
    best_state = None
    best_epoch = 0
    stale_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for points, labels in train_loader:
            points = points.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(points), labels)
            loss.backward()
            optimizer.step()

        val_accuracy = classification_accuracy(model, val_loader)
        print(f"эпоха {epoch:2d} | val accuracy {val_accuracy:.3f}")

        if val_accuracy > best_val + 1e-4:
            best_val = val_accuracy
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= patience:
            break

    model.load_state_dict(best_state)
    test_accuracy = classification_accuracy(model, test_loader)
    print(
        f"лучшая эпоха {best_epoch} | val accuracy {best_val:.3f} | "
        f"test accuracy {test_accuracy:.3f}"
    )
    return {"best_epoch": best_epoch, "val_accuracy": best_val, "test_accuracy": test_accuracy}


pointnet_result = fit_pointnet(pointnet)


### Критические точки

Для каждой компоненты pooled-вектора максимум достигается хотя бы на одной точке. Объединение таких точек называют критическим множеством для данного облака и данной сети.

Задание 3. Найдите индексы критических точек и проверьте, что они воспроизводят pooled-вектор полного облака. Затем сравните их со случайным подмножеством того же размера.


In [ ]:
points, label = test_shapes[3]
points_batch = points.unsqueeze(0).to(device)

pointnet.eval()
with torch.no_grad():
    features = ...
    critical_idx = ...
    pooled_full = features.max(dim=1).values
    pooled_critical = pointnet.point_mlp(
        points_batch[:, critical_idx]
    ).max(dim=1).values



rng = torch.Generator().manual_seed(0)
random_idx = torch.randperm(N_POINTS, generator=rng)[:len(critical_idx)].to(device)

with torch.no_grad():
    pooled_random = pointnet.point_mlp(
        points_batch[:, random_idx]
    ).max(dim=1).values
    logits_full = pointnet.head(pooled_full)
    logits_critical = pointnet.head(pooled_critical)
    logits_random = pointnet.head(pooled_random)

assert torch.allclose(pooled_full, pooled_critical, atol=1e-6)
assert torch.allclose(logits_full, logits_critical, atol=1e-6)

print(f"{SHAPE_NAMES[int(label)]}: {len(critical_idx)} критических точек из {N_POINTS}")
print("изменение logits на критическом множестве:", float((logits_full - logits_critical).norm()))
print("изменение logits на случайном подмножестве:", float((logits_full - logits_random).norm()))

fig, axes = plt.subplots(1, 2, figsize=(9, 4), subplot_kw={"projection": "3d"})
axes[0].scatter(points[:, 0], points[:, 1], points[:, 2], s=3)
axes[0].set_title("все точки")
axes[0].set_axis_off()
critical_points = points[critical_idx.cpu()]
axes[1].scatter(
    critical_points[:, 0], critical_points[:, 1], critical_points[:, 2],
    s=8, c="crimson",
)
axes[1].set_title("критические точки")
axes[1].set_axis_off()
plt.show()


На критическом множестве max-pooled представление совпадает с представлением полного облака. Случайное подмножество того же размера обычно дает другой вектор.

## 3. kNN-представление облака точек

Облако можно представить графом, связав каждую точку с ближайшими соседями. Это другое представление тех же данных, а не следующий слой PointNet. В PointNet++ локальные области строятся иерархически с sampling, grouping и локальными PointNet; один kNN-граф сам по себе не является PointNet++.

Ниже граф только строится и визуализируется. Реализация kNN не является отдельным заданием.


In [ ]:
def knn_graph(points: torch.Tensor, k: int) -> torch.Tensor:
    distances = torch.cdist(points, points)
    distances.fill_diagonal_(float("inf"))
    neighbors = distances.topk(k, largest=False).indices
    centers = torch.arange(points.shape[0]).repeat_interleave(k)
    sources = neighbors.reshape(-1)
    return torch.stack([sources, centers])


points, _ = test_shapes[2]
points = points[:64]
edge_index_knn = knn_graph(points, k=4)

assert edge_index_knn.shape == (2, 64 * 4)
assert (edge_index_knn[0] != edge_index_knn[1]).all()
assert torch.equal(
    torch.bincount(edge_index_knn[1], minlength=64),
    torch.full((64,), 4),
)

fig = plt.figure(figsize=(5, 5))
axis = fig.add_subplot(projection="3d")
axis.scatter(points[:, 0], points[:, 1], points[:, 2], s=12)
for source, target in edge_index_knn.T:
    segment = points[torch.stack([source, target])]
    axis.plot(segment[:, 0], segment[:, 1], segment[:, 2], lw=0.5, c="gray")
axis.set_title("kNN-граф: 4 входящих ребра на точку")
axis.set_axis_off()
plt.show()


## 4. Классификация вершин графа

### Tolokers

В Tolokers вершина соответствует исполнителю на краудсорсинговой платформе. Ребро связывает исполнителей, работавших над одним заданием. По десяти признакам вершины и структуре графа требуется предсказать бинарную метку недобросовестного исполнителя.

Это трансдуктивная классификация вершин: признаки всего графа доступны при обучении, но loss считается только по train-маске.


In [ ]:
from torch_geometric.datasets import HeterophilousGraphDataset
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_remaining_self_loops, degree


dataset = HeterophilousGraphDataset(root="data/Tolokers", name="Tolokers")
graph_data = dataset[0]

positive_share = float(graph_data.y.float().mean())
source, target = graph_data.edge_index
edge_homophily = float((graph_data.y[source] == graph_data.y[target]).float().mean())

print(graph_data)
print("вершин:", graph_data.num_nodes)
print("ребер в edge_index:", graph_data.num_edges)
print("признаков вершины:", dataset.num_features)
print("доля положительного класса:", round(positive_share, 3))
print("доля ребер с одинаковыми метками на концах:", round(edge_homophily, 3))
print("официальных разбиений:", graph_data.train_mask.shape[1])


### Метрика и разбиения

Положительный класс составляет около одной пятой выборки, поэтому accuracy скрывает качество ранжирования редкого класса. Основная метрика - ROC-AUC.

Столбцы `train_mask`, `val_mask` и `test_mask` задают десять официальных разбиений. Это не десять случайных инициализаций одной модели. В основном эксперименте используется split 0; устойчивость к инициализации и к выбору split проверяется отдельно.

Сначала обучим MLP только на признаках вершин. Это baseline для проверки того, дает ли граф дополнительную информацию.


In [ ]:
class NodeMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=32, dropout=0.5):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden_dim)
        self.hidden1 = nn.Linear(hidden_dim, hidden_dim)
        self.hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index=None):
        x = F.relu(self.input(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.hidden1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.hidden2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.output(x).squeeze(-1)


### Канонический GCN

На лекции GCN был записан как усреднение соседей с отдельным преобразованием собственной вершины. Здесь реализуем вариант Kipf-Welling с self-loops и симметричной нормировкой:

$$
H^{(l+1)} = \sigma\!\left(\hat D^{-1/2}\hat A\hat D^{-1/2}H^{(l)}W^{(l)}\right),
\qquad \hat A = A + I.
$$

Для ребра от $j$ к $i$ сообщение имеет вид

$$
m_{j \to i} = \frac{Wh_j}{\sqrt{\hat d_i\hat d_j}}.
$$

Задание 4. Допишите вычисление степеней, коэффициентов нормировки и функцию `message`. Остальная инфраструктура слоя готова.


In [ ]:
class GCNLayer(MessagePassing):
    def __init__(self, in_dim, out_dim):
        super().__init__(aggr="add")
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x, edge_index):
        edge_index, _ = add_remaining_self_loops(
            edge_index, num_nodes=x.size(0)
        )
        source, target = edge_index
        transformed = self.lin(x)
        deg = ...
        inv_sqrt = ...
        norm = ...
        return self.propagate(
            edge_index, x=transformed, norm=norm
        ) + self.bias

    def message(self, x_j, norm):
        return ...


### Проверка реализации на маленьком графе

Посчитаем один слой двумя способами: через `MessagePassing` и умножением на плотную нормализованную матрицу смежности. Затем переставим ребра и проверим, что результат не изменился.


In [ ]:
toy_x = torch.tensor([
    [1.0, 0.0, 0.5],
    [0.0, 1.0, 0.5],
    [1.0, 1.0, 0.0],
    [0.0, 0.5, 1.0],
])
toy_edges = torch.tensor([
    [0, 1, 1, 2, 2, 0, 2, 3],
    [1, 0, 2, 1, 0, 2, 3, 2],
])

set_global_seed(0)
toy_layer = GCNLayer(3, 2)
toy_layer.eval()
sparse_output = toy_layer(toy_x, toy_edges)

edges_with_loops, _ = add_remaining_self_loops(
    toy_edges, num_nodes=toy_x.size(0)
)
source, target = edges_with_loops
adjacency = torch.zeros(toy_x.size(0), toy_x.size(0))
adjacency[target, source] = 1.0
degrees = adjacency.sum(dim=1)
inv_sqrt = degrees.clamp_min(1).pow(-0.5)
normalized_adjacency = inv_sqrt[:, None] * adjacency * inv_sqrt[None, :]
dense_output = normalized_adjacency @ toy_layer.lin(toy_x) + toy_layer.bias

edge_permutation = torch.randperm(toy_edges.size(1))
permuted_output = toy_layer(toy_x, toy_edges[:, edge_permutation])

assert sparse_output.shape == (4, 2)
assert torch.isfinite(sparse_output).all()
assert torch.allclose(sparse_output, dense_output, atol=1e-6)
assert torch.allclose(sparse_output, permuted_output, atol=1e-6)
print("GCNLayer совпадает с плотным вычислением")


In [ ]:
class NodeGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=32, dropout=0.5):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden_dim)
        self.conv1 = GCNLayer(hidden_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.input(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.output(x).squeeze(-1)


### Протокол обучения

MLP и GCN используют одну скрытую размерность, dropout, optimizer и бюджет эпох. GCN отличается наличием агрегации по ребрам. Обе модели возвращают один логит и обучаются с `BCEWithLogitsLoss`.

Early stopping использует только validation ROC-AUC. Лучшее состояние восстанавливается, после чего test ROC-AUC считается один раз.


In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def masked_auc(logits, labels, mask):
    return roc_auc_score(
        labels[mask].detach().cpu().numpy(),
        logits[mask].detach().cpu().numpy(),
    )


def fit_graph_model(
    model,
    data,
    split=0,
    max_epochs=200,
    patience=25,
    lr=0.005,
    weight_decay=5e-4,
    verbose=True,
):
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )

    train_mask = data.train_mask[:, split]
    val_mask = data.val_mask[:, split]
    test_mask = data.test_mask[:, split]

    best_val = -np.inf
    best_epoch = 0
    best_state = None
    stale_epochs = 0
    started = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.binary_cross_entropy_with_logits(
            logits[train_mask], data.y[train_mask].float()
        )
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(data.x, data.edge_index)
            val_auc = masked_auc(val_logits, data.y, val_mask)

        if val_auc > best_val + 1e-4:
            best_val = val_auc
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1

        if verbose and (epoch == 1 or epoch % 20 == 0):
            print(
                f"эпоха {epoch:3d} | loss {loss.item():.4f} | "
                f"val ROC-AUC {val_auc:.4f}"
            )

        if stale_epochs >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        final_logits = model(data.x, data.edge_index)
        final_val_auc = masked_auc(final_logits, data.y, val_mask)
        test_auc = masked_auc(final_logits, data.y, test_mask)

    result = {
        "parameters": count_parameters(model),
        "best_epoch": best_epoch,
        "val_auc": final_val_auc,
        "test_auc": test_auc,
        "seconds": time.perf_counter() - started,
    }
    return model, result


In [ ]:
HIDDEN_DIM = 32
MAIN_SEED = 0
MAIN_SPLIT = 0

set_global_seed(MAIN_SEED)
mlp = NodeMLP(dataset.num_features, hidden_dim=HIDDEN_DIM)
mlp, mlp_result = fit_graph_model(
    mlp, graph_data, split=MAIN_SPLIT
)

set_global_seed(MAIN_SEED)
gcn = NodeGCN(dataset.num_features, hidden_dim=HIDDEN_DIM)
gcn, gcn_result = fit_graph_model(
    gcn, graph_data, split=MAIN_SPLIT
)

main_results = {"MLP": mlp_result, "GCN": gcn_result}

header = f'{"модель":10s} {"параметры":>12s} {"эпоха":>8s} {"val AUC":>10s} {"test AUC":>10s} {"сек":>9s}'
print(header)
print("-" * len(header))
for name, result in main_results.items():
    print(
        f'{name:10s} {result["parameters"]:12d} {result["best_epoch"]:8d} '
        f'{result["val_auc"]:10.4f} {result["test_auc"]:10.4f} '
        f'{result["seconds"]:9.1f}'
    )


### Разбор результатов

Сопоставьте результаты MLP и GCN и ответьте на вопросы.

1. Дала ли структура графа выигрыш относительно MLP на выбранном split?
2. Достаточно ли одного split и одной инициализации для общего вывода о GCN?
3. Почему низкая доля одноименных соседей сама по себе не доказывает бесполезность message passing?
4. Какой вывод можно сделать по этому запуску и что потребуется проверить перед общим выводом?


## Источники

- [S3DIS, Armeni et al.](http://buildingparser.stanford.edu/dataset.html)
- [PointNet, Qi et al.](https://arxiv.org/abs/1612.00593)
- [GCN, Kipf and Welling](https://arxiv.org/abs/1609.02907)
- [GAT, Velickovic et al.](https://arxiv.org/abs/1710.10903)
- [Tolokers и оценка GNN на гетерофильных графах, Platonov et al.](https://arxiv.org/abs/2302.11640)
